# 챕터 09. 예외 처리

## 학습 목표

- 오류(Error)와 예외(Exception)의 차이를 이해한다
- 자주 발생하는 예외 종류를 알고 원인을 파악할 수 있다
- `try / except / else / finally` 구조를 사용해 예외를 처리할 수 있다
- `raise`로 의도적으로 예외를 발생시킬 수 있다
- 사용자 정의 예외 클래스를 만들어 의미 있는 오류 정보를 전달할 수 있다

## 이 챕터를 마치면 할 수 있는 것

- `input()`으로 받은 값을 안전하게 정수로 변환하기
- 파일 읽기 실패(FileNotFoundError)를 처리해 프로그램이 멈추지 않게 하기
- 커스텀 예외 클래스를 설계해 비즈니스 규칙 위반을 명확하게 표현하기

> **선행 챕터**: 챕터 08(입출력)에서 `int(input(...))` 사용 시 발생하는 `ValueError`와  
> 파일 열기 실패 시 발생하는 `FileNotFoundError`를 예고했습니다.  
> 이번 챕터에서 이 두 예외를 포함한 다양한 예외 처리 방법을 본격적으로 다룹니다.

## 1. 오류와 예외의 개념

### 오류(Error) vs 예외(Exception)

| 구분 | 설명 | 처리 가능 여부 |
|------|------|----------------|
| **SyntaxError** | 실행 전 문법 오류. 인터프리터가 코드를 파싱할 수 없음 | 불가 (코드 수정 필요) |
| **Exception** | 실행 중(런타임)에 발생하는 오류 | 가능 (`try/except`로 처리) |

`SyntaxError`는 코드 자체가 잘못된 것이므로 실행조차 되지 않습니다.  
`Exception`은 코드 문법은 올바르지만 실행 중 문제가 생긴 것으로, 프로그래머가 대응 방법을 정할 수 있습니다.

### 자주 만나는 예외 종류

| 예외 | 발생 조건 |
|------|-----------|
| `TypeError` | 자료형 불일치 (`"2" + 2`) |
| `ValueError` | 값은 맞지만 형변환 불가 (`int("abc")`) |
| `IndexError` | 리스트 인덱스 범위 초과 |
| `KeyError` | 딕셔너리에 없는 키 접근 |
| `ZeroDivisionError` | 0으로 나누기 |
| `FileNotFoundError` | 존재하지 않는 파일 열기 |

In [ ]:
# 각 예외가 어떤 상황에서 발생하는지 직접 확인한다
examples = [
    ("TypeError",         lambda: "2" + 2),
    ("ValueError",        lambda: int("abc")),
    ("IndexError",        lambda: [1, 2, 3][10]),
    ("KeyError",          lambda: {"a": 1}["b"]),
    ("ZeroDivisionError", lambda: 10 / 0),
    ("FileNotFoundError", lambda: open("없는파일.txt")),
]

for name, func in examples:
    try:
        func()
    except Exception as e:
        print(f"{name:20s}: {e}")

## 2. try / except 기본 구조

예외가 발생할 수 있는 코드를 `try` 블록 안에 넣고,  
예외가 발생했을 때 실행할 코드를 `except` 블록에 작성합니다.

```python
try:
    # 예외가 발생할 수 있는 코드
except 예외타입 as e:
    # 예외 발생 시 실행
else:
    # 예외가 없을 때만 실행
finally:
    # 항상 실행 (예외 여부 무관)
```

**주요 포인트**

- `as e`: 예외 객체를 변수 `e`에 담아 오류 메시지를 출력하거나 로그에 기록할 수 있다
- 예외 타입을 명시하지 않으면 모든 예외를 잡는다 — 의도치 않은 오류를 숨길 수 있으므로 **가능한 한 구체적인 타입을 명시하는 것을 권장**한다
- `except`는 여러 개 작성할 수 있으며, 위에서부터 순서대로 매칭된다

In [ ]:
# ZeroDivisionError 처리
def 나누기(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("오류: 0으로 나눌 수 없습니다.")
        return None
    return result

print(나누기(10, 2))   # 5.0
print(나누기(10, 0))   # 오류 메시지 출력 후 None 반환

print()


In [ ]:
def 나누기(a, b):
    return a / b

print(나누기(10, 0))   # 오류 메시지 출력 후 None 반환

In [ ]:

# ValueError 처리 — 안전한 input 정수 변환
def 정수_입력(text):
    try:
        value = int(text)
    except ValueError as e:
        print(f"변환 실패: {e}")
        return None
    return value

print(정수_입력("42"))     # 42
print(정수_입력("안녕"))   # 변환 실패 메시지

In [ ]:
# except 여러 개로 예외 타입별 다른 처리
def 파일_첫줄_읽기(경로):
    try:
        with open(경로, "r", encoding="utf-8") as f:
            return f.readline().rstrip()
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {경로}")
    except PermissionError:
        print(f"파일 읽기 권한이 없습니다: {경로}")
    except Exception as e:
        print(f"알 수 없는 오류: {e}")
    return None

print(파일_첫줄_읽기("sample.txt"))    # 챕터 08에서 생성한 파일
print(파일_첫줄_읽기("없는파일.txt"))  # FileNotFoundError 처리

## 3. else / finally

### else 절
`try` 블록이 **예외 없이 끝났을 때만** 실행됩니다.  
성공 후속 처리(결과 저장, 다음 단계 진행 등)를 `try` 블록 밖에서 명확하게 분리할 수 있습니다.

### finally 절
예외 발생 여부와 **관계없이 항상** 실행됩니다.  
파일 닫기, 네트워크 연결 해제 등 반드시 수행해야 하는 정리 작업에 활용합니다.

> **참고**: `with` 문을 사용하면 `finally`에서 파일을 닫는 코드를 직접 작성하지 않아도 됩니다.  
> `with open(...) as f:` 블록을 벗어나면 `__exit__`이 자동으로 호출되어 파일이 닫힙니다.

In [ ]:
def 나누기_전체구조(a, b):
    print(f"--- {a} / {b} 시도 ---")
    try:
        result = a / b
    except ZeroDivisionError:
        print("except: 0으로 나누기 오류 발생")
    else:
        # 예외가 없을 때만 실행
        print(f"else: 성공, 결과 = {result}")
    finally:
        # 예외 여부와 관계없이 항상 실행
        print("finally: 항상 실행됩니다")
    print()

나누기_전체구조(10, 2)
나누기_전체구조(10, 0)

In [ ]:
# 여러 예외를 튜플로 묶어 하나의 except로 처리
def 안전_변환(값):
    """문자열을 정수 또는 실수로 변환한다."""
    try:
        return int(값)
    except ValueError:
        try:
            return float(값)
        except ValueError:
            return None

# TypeError와 ValueError를 하나의 except로 묶기
def 합계(a, b):
    try:
        return a + b
    except (TypeError, ValueError) as e:
        print(f"입력 오류: {e}")
        return None

print(합계(3, 5))       # 8
print(합계("3", 5))     # 입력 오류
print(합계([1], [2]))   # 6 (리스트 연결)

print()
print(안전_변환("42"))    # 42 (int)
print(안전_변환("3.14"))  # 3.14 (float)
print(안전_변환("hello")) # None

## 4. raise — 예외 직접 발생시키기

```python
raise ValueError("메시지")  # 새로운 예외 발생
raise                        # except 블록 안에서 현재 예외를 다시 던짐
```

**언제 사용하나요?**

- 함수 내부에서 유효성 검사를 실패했을 때, 호출한 쪽에 문제를 알려야 할 때
- 잘못된 입력을 조용히 넘기지 않고 명시적으로 오류를 알려야 할 때
- `except`에서 예외를 로그에 기록한 뒤 동일한 예외를 상위로 다시 전달할 때 (`raise` 단독 사용)

> 함수가 `raise`로 예외를 발생시키면, 함수를 호출한 코드에서 `try/except`로 처리할 수 있습니다.

In [ ]:
def 나이_검증(나이):
    """나이가 유효한 범위인지 확인한다."""
    if not isinstance(나이, int):
        raise TypeError(f"나이는 정수여야 합니다. (입력값: {type(나이).__name__})")
    if 나이 < 0 or 나이 > 150:
        raise ValueError(f"나이는 0~150 사이여야 합니다. (입력값: {나이})")
    return True

# 정상 입력
try:
    나이_검증(25)
    print("유효한 나이입니다.")
except (TypeError, ValueError) as e:
    print(f"오류: {e}")

print()

# 다양한 입력값 테스트
for 테스트값 in [25, -1, 200, "서른"]:
    try:
        나이_검증(테스트값)
        print(f"{테스트값!r}: 유효")
    except (TypeError, ValueError) as e:
        print(f"{테스트값!r}: {e}")

## 5. 사용자 정의 예외 클래스

`Exception`을 상속받아 원하는 이름의 예외 클래스를 만들 수 있습니다.

```python
class 내예외(Exception):
    pass
```

**왜 사용하나요?**

- `ValueError`보다 `잔액부족오류`가 무슨 문제인지 더 명확하게 전달한다
- `except 잔액부족오류`로 정확히 그 오류만 잡을 수 있어 다른 예외와 섞이지 않는다
- `__init__`으로 잔액, 출금액 같은 추가 정보를 예외 객체에 담을 수 있다

> 클래스 이름은 `오류`, `에러`, `Error`, `Exception` 등으로 끝내는 것이 관례입니다.

In [ ]:
class 잔액부족오류(Exception):
    """출금액이 잔액을 초과할 때 발생하는 예외"""
    def __init__(self, 잔액, 출금액):
        self.잔액 = 잔액
        self.출금액 = 출금액
        super().__init__(f"잔액 부족: 잔액 {잔액}원, 출금 요청 {출금액}원")

class 계좌:
    def __init__(self, 잔액):
        self.잔액 = 잔액

    def 출금(self, 금액):
        if 금액 > self.잔액:
            raise 잔액부족오류(self.잔액, 금액)
        self.잔액 -= 금액
        return 금액

내계좌 = 계좌(10000)

try:
    내계좌.출금(3000)
    print(f"3000원 출금 완료. 잔액: {내계좌.잔액}원")
    내계좌.출금(9000)   # 잔액 부족
except 잔액부족오류 as e:
    print(f"출금 실패: {e}")
    print(f"  현재 잔액: {e.잔액}원, 요청 금액: {e.출금액}원")

In [ ]:
# 이 셀을 실행하면 입력창이 나타납니다
def 정수_입력받기(안내문구, 최솟값=None, 최댓값=None):
    """올바른 정수를 입력받을 때까지 반복한다."""
    while True:
        try:
            값 = int(input(안내문구))
        except ValueError:
            print("숫자만 입력하세요.")
            continue

        if 최솟값 is not None and 값 < 최솟값:
            print(f"{최솟값} 이상의 값을 입력하세요.")
            continue
        if 최댓값 is not None and 값 > 최댓값:
            print(f"{최댓값} 이하의 값을 입력하세요.")
            continue

        return 값

나이 = 정수_입력받기("나이를 입력하세요 (1~120): ", 최솟값=1, 최댓값=120)
print(f"입력한 나이: {나이}세")

In [ ]:
def 파일_읽기(경로):
    """파일을 읽어 내용을 반환한다. 실패 시 None."""
    try:
        with open(경로, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"[오류] 파일 없음: {경로}")
        return None
    except UnicodeDecodeError:
        print(f"[오류] 인코딩 오류: {경로}")
        return None

# 존재하는 파일 (챕터 08에서 생성)
내용 = 파일_읽기("members.csv")
if 내용:
    print("=== members.csv ===")
    print(내용)

# 존재하지 않는 파일
파일_읽기("없는파일.txt")

In [ ]:
# 종합 실습 — 예외 처리가 적용된 사칙연산 계산기
def 계산(a, b, 연산자):
    """두 수와 연산자를 받아 계산 결과를 반환한다."""
    try:
        a, b = float(a), float(b)
    except ValueError:
        raise ValueError("숫자로 변환할 수 없는 입력입니다.")

    if 연산자 == "+":
        return a + b
    elif 연산자 == "-":
        return a - b
    elif 연산자 == "*":
        return a * b
    elif 연산자 == "/":
        if b == 0:
            raise ZeroDivisionError("0으로 나눌 수 없습니다.")
        return a / b
    else:
        raise ValueError(f"지원하지 않는 연산자: {연산자}")

# 테스트
테스트케이스 = [
    (10, 3,     "+"),
    (10, 3,     "-"),
    (10, 3,     "*"),
    (10, 3,     "/"),
    (10, 0,     "/"),      # ZeroDivisionError
    ("abc", 3,  "+"),     # ValueError
    (10, 3,     "%"),      # ValueError (지원 안함)
]

for a, b, op in 테스트케이스:
    try:
        결과 = 계산(a, b, op)
        print(f"{a} {op} {b} = {결과}")
    except (ZeroDivisionError, ValueError) as e:
        print(f"[오류] {a} {op} {b}: {e}")

## 챕터 정리

이번 챕터에서 배운 내용을 정리합니다.

- **오류 vs 예외**: `SyntaxError`는 코드 수정이 필요하고, `Exception`은 `try/except`로 처리 가능
- **주요 예외**: `TypeError`, `ValueError`, `IndexError`, `KeyError`, `ZeroDivisionError`, `FileNotFoundError`
- **try/except**: 예외가 발생할 수 있는 코드를 감싸고, 예외 타입을 명시해 처리
- **except 여러 개**: 예외 타입마다 다른 대응을 할 수 있다
- **except (A, B)**: 여러 예외를 하나의 블록으로 묶어 처리
- **else**: 예외가 없을 때만 실행 — 성공 후속 처리에 적합
- **finally**: 예외 여부와 무관하게 항상 실행 — 리소스 정리에 활용
- **raise**: 유효성 검사 실패 시 의도적으로 예외를 발생시켜 호출자에게 알림
- **사용자 정의 예외**: `Exception` 상속으로 의미 있는 예외 클래스 설계 가능

### try / except / else / finally 전체 구조

| 절 | 실행 조건 | 주요 용도 |
|----|-----------|-----------|
| `try` | 항상 실행 시도 | 예외 가능성 있는 코드 |
| `except` | 예외 발생 시 | 예외 유형별 처리 |
| `else` | 예외 없을 때만 | 성공 후속 처리 |
| `finally` | 항상 | 리소스 정리 |

---

**다음 챕터 예고**: 챕터 10 — 모듈과 패키지

`import`로 외부 코드를 불러오고, `pip`로 패키지를 설치하며, 파이썬 표준 라이브러리를 활용하는 방법을 학습합니다.